<a href="https://colab.research.google.com/github/aditidandge/Artificial_Intelligence_Lab_SE_B_15/blob/master/Prac_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Uninformed searching techniques

### Breadth-First Search (BFS)

BFS explores all the neighbor nodes at the present depth prior to moving on to the nodes at the next depth level. It typically uses a queue data structure.

In [ ]:
from collections import deque

def bfs(graph, start_node):
    visited = set()
    queue = deque([start_node])
    visited.add(start_node)

    traversal_order = []

    while queue:
        current_node = queue.popleft()
        traversal_order.append(current_node)

        for neighbor in graph[current_node]:
            if neighbor not in visited:
                visited.add(neighbor)
                queue.append(neighbor)

    return traversal_order

# Example Graph
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

print("BFS Traversal starting from 'A':", bfs(graph, 'A'))

BFS Traversal starting from 'A': ['A', 'B', 'C', 'D', 'E', 'F']


### Depth-First Search (DFS)

DFS explores as far as possible along each branch before backtracking. It typically uses recursion or a stack data structure.

In [ ]:
def dfs(graph, start_node, visited=None):
    if visited is None:
        visited = set()

    traversal_order = []

    if start_node not in visited:
        visited.add(start_node)
        traversal_order.append(start_node)

        for neighbor in graph[start_node]:
            traversal_order.extend(dfs(graph, neighbor, visited)) # Extend with results from recursive calls

    return traversal_order

# Example Graph (using the same graph as BFS for consistency)
graph = {
    'A': ['B', 'C'],
    'B': ['A', 'D', 'E'],
    'C': ['A', 'F'],
    'D': ['B'],
    'E': ['B', 'F'],
    'F': ['C', 'E']
}

print("DFS Traversal starting from 'A':", dfs(graph, 'A'))

DFS Traversal starting from 'A': ['A', 'B', 'D', 'E', 'F', 'C']


### A* Search Algorithm

A* is an informed search algorithm that uses a heuristic function to guide its search. It combines features of Dijkstra's algorithm and Greedy Best-First Search to find the shortest path from a start node to a goal node.

In [3]:
import heapq # For priority queue implementation

def a_star_search(graph, start, goal, heuristic):
    # The priority queue stores tuples: (f_cost, current_cost, current_node, path)
    # f_cost = current_cost + heuristic(current_node, goal)
    priority_queue = [(0 + heuristic(start, goal), 0, start, [start])]
    visited_costs = {start: 0} # Stores the g_cost (actual cost from start) for visited nodes

    while priority_queue:
        f_cost, current_cost, current_node, path = heapq.heappop(priority_queue)

        if current_node == goal:
            return path, current_cost

        for neighbor, edge_weight in graph[current_node].items():
            new_cost = current_cost + edge_weight

            # If this path is better than any previously found path to the neighbor
            if neighbor not in visited_costs or new_cost < visited_costs[neighbor]:
                visited_costs[neighbor] = new_cost
                new_f_cost = new_cost + heuristic(neighbor, goal)
                heapq.heappush(priority_queue, (new_f_cost, new_cost, neighbor, path + [neighbor]))

    return None, float('inf') # Goal not reachable

# Example Graph (Adjacency list with weights)
graph = {
    'A': {'B': 1, 'C': 3},
    'B': {'A': 1, 'D': 3, 'E': 6},
    'C': {'A': 3, 'F': 2},
    'D': {'B': 3},
    'E': {'B': 6, 'F': 1},
    'F': {'C': 2, 'E': 1}
}

# Heuristic function (e.g., straight-line distance, or simply a placeholder for this example)
# In a real-world scenario, this would estimate the cost from current_node to goal.
# For this example, we'll use a simple heuristic that returns 0 for all nodes to demonstrate.
# A more complex example would require actual coordinates or problem-specific estimates.

def heuristic(node, goal):
    # For demonstration, let's assume some arbitrary heuristic values.
    # In a real problem, this would be problem-specific (e.g., Euclidean distance).
    h_values = {
        'A': 6,
        'B': 4,
        'C': 2,
        'D': 4,
        'E': 1,
        'F': 0   # Goal node heuristic is typically 0
    }
    return h_values.get(node, 0) # Return 0 if node not in h_values

start_node = 'A'
goal_node = 'F'

path, cost = a_star_search(graph, start_node, goal_node, heuristic)

if path:
    print(f"A* Path from {start_node} to {goal_node}: {path}")
    print(f"Total Cost: {cost}")
else:
    print(f"No path found from {start_node} to {goal_node}")

A* Path from A to F: ['A', 'C', 'F']
Total Cost: 5


### Minimax Algorithm

The Minimax algorithm is a decision-making algorithm used in artificial intelligence, game theory, and other fields. It is primarily used for two-player zero-sum games, meaning that one player's gain is equivalent to the other player's loss.

**Key Concepts:**
- **Maximizing Player:** Aims to get the highest possible score.
- **Minimizing Player:** Aims to get the lowest possible score.
- **Game Tree:** A tree structure where each node represents a state of the game, and edges represent possible moves.
- **Terminal Nodes (Leaf Nodes):** Nodes at the end of the game tree, which have a specific score (outcome) for the maximizing player.

**How it works:**
TheMinimax algorithm works by recursively exploring the game tree from the current state down to a certain depth (or until terminal nodes are reached). It then assigns values to the nodes based on the scores of the terminal nodes, assuming both players play optimally.

- The **Maximizing Player** chooses the move that leads to the highest possible score.
- The **Minimizing Player** chooses the move that leads to the lowest possible score.

In [4]:
def minimax(node_id, depth, maximizing_player, game_tree):
    """
    Implements the Minimax algorithm.

    Args:
        node_id (str): The current node's identifier in the game tree.
        depth (int): The current depth of the search (starts at 0 for the root).
        maximizing_player (bool): True if it's the maximizing player's turn, False otherwise.
        game_tree (dict): A dictionary representing the game tree.
                          Keys are node IDs. Values are either an integer (if terminal node)
                          or a list of child node IDs (if non-terminal node).

    Returns:
        tuple: A tuple containing the optimal value and the best move (child node ID)
               from the current node. If it's a terminal node, returns (value, None).
    """

    # Base case: If it's a terminal node, return its value and no move.
    if isinstance(game_tree[node_id], int):
        return game_tree[node_id], None

    if maximizing_player:
        max_eval = -float('inf')
        best_move = None
        for child_id in game_tree[node_id]:
            # Recursively call minimax for the minimizing player
            eval, _ = minimax(child_id, depth + 1, False, game_tree)
            if eval > max_eval:
                max_eval = eval
                best_move = child_id # Store the move that leads to this max_eval
        return max_eval, best_move
    else:  # Minimizing player
        min_eval = float('inf')
        best_move = None
        for child_id in game_tree[node_id]:
            # Recursively call minimax for the maximizing player
            eval, _ = minimax(child_id, depth + 1, True, game_tree)
            if eval < min_eval:
                min_eval = eval
                best_move = child_id # Store the move that leads to this min_eval
        return min_eval, best_move

# Example Game Tree:
# This tree represents a simple game where players take turns moving down.
# The root 'A' is the maximizing player's turn.
# Nodes 'B' and 'C' are the minimizing player's turn.
# Nodes 'D', 'E', 'F', 'G' are terminal nodes with associated scores.
#
#       MAX (A)
#      /     \
#    MIN (B) MIN (C)
#   /   \   /   \
#  MAX(D) MAX(E) MAX(F) MAX(G) - Terminal nodes with scores
#   (3)   (12)    (8)    (2)

example_game_tree = {
    'A': ['B', 'C'],
    'B': ['D', 'E'],
    'C': ['F', 'G'],
    'D': 3,
    'E': 12,
    'F': 8,
    'G': 2
}

# Start the Minimax algorithm from the root node 'A'.
# Initial call: depth = 0, maximizing_player = True (Player A is the maximizing player).
optimal_value, best_first_move = minimax('A', 0, True, example_game_tree)

print(f"The optimal value for the root node 'A' using Minimax is: {optimal_value}")
print(f"The best first move for the maximizing player from 'A' is: {best_first_move}")

The optimal value for the root node 'A' using Minimax is: 3
The best first move for the maximizing player from 'A' is: B
